# 咖啡烘焙分类：TensorFlow.js 浏览器版

对应原课程的「CoffeeRoasting_TF」实验。保持 NumPy 数据处理、神经网络分类与 Matplotlib 可视化，实际训练由浏览器内的 TensorFlow.js 完成。

本实验采用 `await model.fit(...)` 调用异步训练接口，既不需要 GPU 服务器，也不需要 Python TensorFlow。

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from browser_ml.tfjs_bridge import load_tfjs, Dense, Sequential, backend

tfjs = await load_tfjs()
print("TensorFlow.js:", tfjs.version.tfjs, "| 后端:", backend())

## 准备课程数据

与原实验一样，根据烘焙温度和持续时间判断是否合格。

In [ ]:
rng = np.random.default_rng(2)
X = rng.random(400).reshape(-1, 2)
X[:, 1] = X[:, 1] * 4 + 11.5
X[:, 0] = X[:, 0] * 135 + 150
Y = np.asarray([
    int(175 < t < 260 and 12 < d < 15 and d <= -3 / 85 * t + 21)
    for t, d in X
], dtype="float32").reshape(-1, 1)
print("样本数量", len(X), "合格", int(Y.sum()))
fig, ax = plt.subplots()
ax.scatter(X[Y.ravel()==0,0], X[Y.ravel()==0,1], alpha=.4, label="不合格")
ax.scatter(X[Y.ravel()==1,0], X[Y.ravel()==1,1], alpha=.7, label="合格")
ax.legend()
plt.show()

## 数据归一化与模型训练

使用两层全连接网络。通过 Python 桥接调用 TF.js 的 `tf.sequential`、`tf.layers.dense` 和 `model.fit`。

In [ ]:
mean = X.mean(axis=0)
std = X.std(axis=0)
Xn = ((X - mean) / std).astype("float32")
model = Sequential([
    Dense(6, activation="sigmoid", input_shape=(2,)),
    Dense(1, activation="sigmoid"),
])
model.compile(loss="binaryCrossentropy", learning_rate=0.01)
history = await model.fit(Xn, Y, epochs=70, batch_size=32)
pred = model.predict(Xn)
print("训练准确率:", round(float(np.mean((pred > 0.5) == Y)), 3))

## 观察损失和预测结果

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["loss"])
axes[0].set(title="训练损失", xlabel="Epoch")
x_test = np.asarray([[200, 13.9], [200, 17]], dtype="float32")
print("两组烘焙测试数据的预测概率:", model.predict((x_test-mean)/std).ravel())
xx, yy = np.meshgrid(np.linspace(150, 285, 45), np.linspace(11.5, 15.5, 45))
grid = np.stack([xx.ravel(), yy.ravel()], axis=1)
prob = model.predict((grid-mean)/std).reshape(xx.shape)
axes[1].contourf(xx, yy, prob, levels=np.linspace(0, 1, 11), cmap="RdBu_r")
axes[1].set(xlabel="温度", ylabel="持续时间", title="预测概率")
plt.tight_layout()
plt.show()

训练结束后如需回收 WebAssembly / TensorFlow.js 张量内存，可以运行 `model.dispose()`。